# Week 3 — One Search Function, Four Algorithms

**Lesson plan:** [`../weeks/week-03.md`](../weeks/week-03.md) · **Slides:** [`../slides/week-03/deck.md`](../slides/week-03/deck.md)

Live coding, 25 minutes — the highest-value 25 minutes of the week, because it
turns four algorithms into **one idea plus a frontier discipline**.

> ⚠️ **The `search()` you write here is reused in week 4 (A\* is a 10-line diff)
> and week 9 (the forward planner).** If it is broken, it stays broken until week 9.
> Fix it now.

## 1 · The Problem interface

Everything in weeks 3–9 talks to these five methods. This is the *five-component
problem formulation* from the slides, as code.

In [ ]:
from collections import deque
import heapq
import itertools


class Problem:
    """The five components: initial state, actions, transition, goal test, cost."""

    def __init__(self, initial, goal=None):
        self.initial, self.goal = initial, goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return state == self.goal

    def step_cost(self, state, action):
        return 1


class Node:
    __slots__ = ("state", "parent", "action", "g")

    def __init__(self, state, parent=None, action=None, g=0):
        self.state, self.parent, self.action, self.g = state, parent, action, g

    def path(self):
        node, out = self, []
        while node.parent is not None:
            out.append(node.action)
            node = node.parent
        return out[::-1]

    def __repr__(self):
        return f"Node({self.state}, g={self.g})"

## 2 · The frontier — the *only* thing that differs between BFS, DFS, and UCS

Three data structures: FIFO queue, LIFO stack, priority queue. Same search loop.
This is the whole insight of the week.

In [ ]:
class Frontier:
    """FIFO -> BFS.  LIFO -> DFS.  Priority by g -> UCS."""

    def __init__(self, kind):
        self.kind = kind
        self.counter = itertools.count()
        if kind in ("fifo", "lifo"):
            self.data = deque()
        elif kind == "priority":
            self.data = []
        else:
            raise ValueError(kind)

    def push(self, node):
        if self.kind == "priority":
            heapq.heappush(self.data, (node.g, next(self.counter), node))
        else:
            self.data.append(node)

    def pop(self):
        if self.kind == "priority":
            return heapq.heappop(self.data)[2]
        if self.kind == "fifo":
            return self.data.popleft()
        return self.data.pop()

    def __len__(self):
        return len(self.data)

## 3 · The generic search

Two details in this function are load-bearing. Both are annotated. Both will show
up as bugs in Duel 1 if you skip them.

In [ ]:
def search(problem, frontier_kind):
    start = Node(problem.initial)
    if problem.is_goal(start.state):
        return start, 0

    frontier = Frontier(frontier_kind)
    frontier.push(start)
    explored = set()
    expansions = 0                       # instrument from line one

    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):  # goal test on EXPANSION, not generation
            return node, expansions
        if node.state in explored:
            continue
        explored.add(node.state)
        expansions += 1
        for action in problem.actions(node.state):
            child_state = problem.result(node.state, action)
            if child_state not in explored:
                cost = problem.step_cost(node.state, action)
                frontier.push(Node(child_state, node, action, node.g + cost))
    return None, expansions

## 4 · ⚠️ Teaching moment 1: goal test on expansion, not generation

*The* classic bug. Here is the three-node graph that exposes it:

```
start --(10)--> A
start --(1)---> B --(1)--> A
```

The optimal path to A costs **2**. A UCS that tests for the goal when a node is
*generated* returns **10**, because `start -> A` is generated first.

Run it. Then remember it when you write Duel 1.

In [ ]:
class TinyGraph(Problem):
    EDGES = {"start": [("A", 10), ("B", 1)], "B": [("A", 1)], "A": []}

    def actions(self, state):
        return [dst for dst, _ in self.EDGES[state]]

    def result(self, state, action):
        return action

    def step_cost(self, state, action):
        return dict(self.EDGES[state])[action]


p = TinyGraph("start", "A")

node, exp = search(p, "priority")
print(f"CORRECT (test on expansion): path={node.path()}  cost={node.g}")


def search_buggy(problem, frontier_kind):
    """Identical, except the goal test happens at GENERATION time."""
    start = Node(problem.initial)
    frontier = Frontier(frontier_kind)
    frontier.push(start)
    explored = set()
    while frontier:
        node = frontier.pop()
        if node.state in explored:
            continue
        explored.add(node.state)
        for action in problem.actions(node.state):
            s2 = problem.result(node.state, action)
            child = Node(s2, node, action,
                         node.g + problem.step_cost(node.state, action))
            if problem.is_goal(s2):          # <-- BUG: tests too early
                return child, 0
            frontier.push(child)
    return None, 0


bad, _ = search_buggy(p, "priority")
print(f"BUGGY   (test on generation): path={bad.path()}  cost={bad.g}   <-- not optimal")
assert node.g == 2 and bad.g == 10
print("\nBoth assertions hold. UCS optimality depends on WHERE you test.")

## 5 · The 8-puzzle

State is a 9-tuple; `0` is the blank. This is the *problem formulation is a design
decision* slide, made concrete — we chose a flat tuple so states are hashable and
comparable in one operation.

In [ ]:
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


class EightPuzzle(Problem):
    MOVES = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5),
             3: (0, 4, 6), 4: (1, 3, 5, 7), 5: (2, 4, 8),
             6: (3, 7), 7: (4, 6, 8), 8: (5, 7)}

    def __init__(self, initial, goal=GOAL):
        super().__init__(initial, goal)

    def actions(self, state):
        return self.MOVES[state.index(0)]

    def result(self, state, action):
        s = list(state)
        b = state.index(0)
        s[b], s[action] = s[action], s[b]
        return tuple(s)


def show(state):
    for r in range(0, 9, 3):
        print(" ".join("_" if v == 0 else str(v) for v in state[r:r + 3]))


# Depths below are VERIFIED by exhaustive BFS from the goal (see the appendix
# cell at the bottom). Do not trust hand-labelled puzzle depths -- including mine.
D2  = (1, 2, 3, 4, 5, 6, 0, 7, 8)      # optimal 2
D8  = (0, 4, 2, 5, 1, 3, 7, 8, 6)      # optimal 8
D12 = (5, 4, 2, 7, 0, 3, 8, 1, 6)      # optimal 12
D16 = (7, 5, 2, 4, 0, 3, 8, 1, 6)      # optimal 16  <- week 4 uses this one

show(D12)

## 6 · Iterative deepening

DFS's memory with BFS's completeness. The claim to test: the repeated work is a
**constant factor**, not an exponential one, because the deepest level dominates
the sum.

In [ ]:
def depth_limited(problem, limit):
    """Tree search to a fixed depth. Returns (node, expansions, hit_limit)."""
    counter = [0]

    def recurse(node, depth):
        if problem.is_goal(node.state):
            return node, False
        if depth == 0:
            return None, True
        counter[0] += 1
        cutoff = False
        for action in problem.actions(node.state):
            s2 = problem.result(node.state, action)
            # skip the immediate parent -- cheap 2-cycle avoidance, no memory cost
            if node.parent is not None and s2 == node.parent.state:
                continue
            child = Node(s2, node, action, node.g + 1)
            result, hit = recurse(child, depth - 1)
            if result is not None:
                return result, False
            cutoff = cutoff or hit
        return None, cutoff

    node, hit = recurse(Node(problem.initial), limit)
    return node, counter[0], hit


def ids(problem, max_depth=40):
    total = 0
    for limit in range(max_depth + 1):
        node, exp, hit = depth_limited(problem, limit)
        total += exp
        if node is not None:
            return node, total
        if not hit:
            return None, total      # exhausted the whole space, no solution
    return None, total


node, exp = ids(EightPuzzle(D8))
print(f"IDS: {len(node.path())} moves, {exp:,} expansions")

## 7 · The comparison table — let the numbers do the talking

In [ ]:
import time

instances = [("depth 2", D2), ("depth 8", D8), ("depth 12", D12)]

for label, start in instances:
    p = EightPuzzle(start)
    print(f"\n{label}")
    print(f"  {'algorithm':<10}{'expansions':>13}{'sol.length':>12}{'optimal?':>10}{'time':>9}")
    print("  " + "-" * 54)
    rows = []
    for name, kind in [("BFS", "fifo"), ("DFS", "lifo"), ("UCS", "priority")]:
        t0 = time.perf_counter()
        node, exp = search(p, kind)
        dt = time.perf_counter() - t0
        rows.append((name, exp, len(node.path()), dt))
    t0 = time.perf_counter()
    node, exp = ids(p)
    rows.append(("IDS", exp, len(node.path()), time.perf_counter() - t0))

    best = min(r[2] for r in rows)
    for name, exp, n, dt in rows:
        ok = "yes" if n == best else "NO"
        print(f"  {name:<10}{exp:>13,}{n:>12}{ok:>10}{dt:>8.3f}s")

### Read the table out loud

- **DFS is not optimal, and the table is brutal about it.** On the depth-12
  instance it returns a solution hundreds of moves long. It *found* a goal; it
  found a terrible one. Optimality is not a nice-to-have you can eyeball.
- **BFS and UCS agree exactly** here, because every step costs 1. On `TinyGraph`
  they would not — that difference is the entire reason UCS exists.
- **IDS tracks BFS closely** in expansions while using O(bd) memory instead of
  O(b^d). Compute `IDS / BFS` below and see how small the overhead really is.

> **Instrument from line one.** `expansions` is not decoration — it is the only
> currency in which these algorithms can be compared, and it is scorecard axis 3.
> **Never write a search without a counter.**

In [ ]:
p = EightPuzzle(D12)
_, bfs_exp = search(p, "fifo")
_, ids_exp = ids(p)
print(f"BFS expansions : {bfs_exp:,}")
print(f"IDS expansions : {ids_exp:,}")
print(f"overhead ratio : {ids_exp / bfs_exp:.2f}x")
print("""
The textbook figure for the overhead of IDS over BFS is about 11% for a
branching factor near 10. The 8-puzzle's effective branching factor is smaller
(~1.7 after the parent check), so your ratio will differ -- and may even favour
IDS, because IDS does TREE search and revisits states BFS stores.

The point is the ORDER OF MAGNITUDE: a constant factor, not an exponential one.
You bought a huge memory saving for a small, bounded amount of repeated work.""")

## 8 · Appendix — verify the depths yourself

Never trust a hand-labelled puzzle instance, including the ones in this notebook.
This cell does an exhaustive BFS backwards from the goal and prints the true
optimal depth of every instance used above.

It takes a few seconds and touches all 181 440 reachable states — which is also a
useful reminder of how small the 8-puzzle is, and how hopeless the 15-puzzle would
be for these methods.

In [ ]:
from collections import deque as _dq

def true_depths():
    dist = {GOAL: 0}
    q = _dq([GOAL])
    pz = EightPuzzle(GOAL)
    while q:
        s = q.popleft()
        for a in pz.actions(s):
            t = pz.result(s, a)
            if t not in dist:
                dist[t] = dist[s] + 1
                q.append(t)
    return dist

dist = true_depths()
print(f"reachable states: {len(dist):,}   hardest instance: {max(dist.values())} moves\n")
for name, s in [("D2", D2), ("D8", D8), ("D12", D12), ("D16", D16)]:
    print(f"  {name:<4} {s}  true optimal depth = {dist[s]}")

assert dist[D2] == 2 and dist[D8] == 8 and dist[D12] == 12 and dist[D16] == 16
print("\nAll four labels verified.")

## 9 · Save this for later

Week 4 imports `Problem`, `Node`, `search`, `EightPuzzle`, `GOAL`, and `D16` and
adds **ten lines** to get A\*. Week 9 reuses `search` unmodified for a STRIPS
planner.

Copy this notebook's classes into `aicourse/search.py` in your repo now — the
studio and Duel 1 both expect them there.